<a href="https://colab.research.google.com/github/hasan2733/ml_practice/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasan2733/ml_practice/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames a decision before choosing or training a model. I am using the small anonymized starter dataset in this repository. The unit of analysis is one content page, represented by one row.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (predefined lane).**

I chose this lane because the starter dataset contains page-level search and engagement signals, content-age information, and an existing trend label. An editor cannot manually investigate every page at once, so a ranked review queue could help decide which pages to inspect first. This is a provisional choice for the next seven weeks, not a claim that a model will automatically improve content performance. I will compare a simple, explainable rule-based baseline with a scoring approach and check whether the ranking is useful on held-out clients.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Given the observable search, engagement, and content-age signals in the starter dataset, can I rank content pages so that an editor can identify pages worth reviewing first?

- **Decision:** Which pages should receive limited editorial review time first?
- **Who acts and what they do:** An editor reviews the highest-ranked pages, checks the page and its context, and decides whether to update it, investigate its click-through rate or engagement, or leave it unchanged.
- **Output:** A ranked queue with a priority score and human-readable reasons. The score is decision support, not an automatic publishing instruction.
- **Cost of a wrong call:** A false positive wastes editorial time on a page that did not need attention. A false negative can leave a potentially useful improvement opportunity undiscovered. I expect both costs to matter; I will measure top-of-queue usefulness rather than relying only on overall accuracy.
- **Why data or ML might help:** Several signals may interact, and their usefulness may differ by page or client. A fixed rule may still be sufficient, so ML must earn its place by outperforming a transparent baseline on held-out clients.
- **Task type and primary metric:** Ranking/scoring; precision@K (starting with precision@50) measures how many pages in the top K are labelled as declining. I will also report the label prevalence and compare with a simple baseline.

## 3. Quick look at the data (2–3 real numbers)

The code below loads the repository's starter CSV when available; otherwise it tries the public raw file from this repository. It calculates the supporting numbers directly from the data rather than typing them into the calculation.

The starter data dictionary defines one row per pseudonymized content item and says there are 32 pseudonymized clients. Its decline label is defined as `trend_direction == "down"`. That label describes an observed trend in the snapshot; it is not proof that a page will decline in the future or that refreshing it will improve performance.

In [1]:
from pathlib import Path
import pandas as pd

relative_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/ml_practice/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in relative_candidates if p.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
else:
    raw_url = (
        "https://raw.githubusercontent.com/hasan2733/ml_practice/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(raw_url)

required_columns = {"content_id", "client_id", "trend_direction"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Starter dataset is missing required columns: {sorted(missing_columns)}")

declining_count = int(df["trend_direction"].astype(str).str.lower().eq("down").sum())
declining_rate = declining_count / len(df) if len(df) else 0.0
unique_clients = int(df["client_id"].nunique())

print(f"Content pages (rows): {len(df):,}")
print(f"Distinct pseudonymized clients: {unique_clients:,}")
print(f"Pages labelled as declining: {declining_count:,} ({declining_rate:.1%})")

Content pages (rows): 30,000
Distinct pseudonymized clients: 32
Pages labelled as declining: 16,262 (54.2%)


**What these numbers suggest:** The dataset covers 30,000 pages across 32 pseudonymized clients, and 16,262 pages (54.2%) carry the snapshot's declining label. This is enough volume to investigate whether a short review queue can prioritize labelled pages better than a simple baseline. The label prevalence is not evidence that a refresh will work, and the dataset's client structure means validation should hold out clients rather than randomly splitting individual rows.

## 4. Careful words: what I can and can't claim

**I can claim:** The starter snapshot contains the page-level signals and label distribution measured above. If evaluation supports it, I can report that a ranking method places more currently labelled declining pages near the top of a held-out review queue than a stated baseline.

**I cannot claim:** The label is not a future outcome; it is derived from the observed trend direction. I cannot claim that the model predicts future decline, that an editorial refresh causes better traffic, or that it predicts or explains Google's ranking algorithm. I also cannot generalize from this anonymized starter sample to every website or client.

**Important data safeguards:** `trend_direction` defines the label and must not be used as a model feature; `trend_pct` is also part of the trend-label construction and must not be used as a feature. Pseudonymous IDs are for grouping and client-held-out validation only, never predictive features. Rate columns such as `ctr` are expressed as percentages multiplied by 100 (so 0.76 means 0.76%, not 76%).

## 5. Self-check

- [x] I chose one predefined lane and explained why it is worth investigating.
- [x] I named the decision, the person taking action, the output, and the cost of wrong calls.
- [x] The code loads the starter dataset and calculates three supporting statistics.
- [x] I distinguish the observed trend label from future outcomes and causal claims.
- [x] I identified leakage risks and the need for client-held-out validation.
- [ ] Run all cells in the notebook environment you will submit from and confirm there are no errors.
- [ ] Commit the executed notebook to `work/notebooks/w01_research_question.ipynb`, then submit the repository URL on the assignment card.

The lane remains provisional and can be changed by the end of Week 4.

In [2]:
# Basic consistency checks for the framing numbers.
assert len(df) == 30_000, f"Expected 30,000 starter rows, got {len(df):,}"
assert unique_clients == 32, f"Expected 32 pseudonymized clients, got {unique_clients:,}"
assert declining_count == 16_262, f"Unexpected declining-label count: {declining_count:,}"
assert "trend_direction" in df.columns
print("Self-check passed: row count, client count, and declining-label count match the starter release.")

Self-check passed: row count, client count, and declining-label count match the starter release.
